In [1]:
import os
from dotenv import load_dotenv

print("Thư mục hiện tại:", os.getcwd())

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    print("Đã chuyển thư mục ra root:", os.getcwd())

load_dotenv(override=True)
print("Đã load file .env thành công!")

Thư mục hiện tại: /home/huyy-giaa/Develop/VietCulture_Rag/notebooks
Đã chuyển thư mục ra root: /home/huyy-giaa/Develop/VietCulture_Rag
Đã load file .env thành công!


In [2]:
import json
import pandas as pd
from tqdm import tqdm
import time
from src.agent.graph import create_agent_bundle, invoke_agent 
from src.retrieval import qa_retriever
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
bundle = create_agent_bundle()

with open("notebooks/output_benchmark.json") as f:
    data = json.load(f)

results = []

for sample in tqdm(data[:3], desc="Đang đánh giá"): 
    for q in sample["user_query"]:

        user_query = q["text"]
        query_id = q["query_id"]

        conv_id = f"eval_{sample.get('benchmark_id', '0')}_{query_id}" 
    
        retrieved_docs = bundle.retriever.retrieve(query=user_query)
        retrieved_texts = [chunk.document.page_content for chunk in retrieved_docs]
        retrieved_ids = [chunk.document.metadata.get("doc_id", "") for chunk in retrieved_docs]
    
        print(f"\nĐang xử lý: {user_query}") 
        
        graph_response = invoke_agent(
            bundle=bundle,
            user_id="evaluator_user", 
            conversation_id=conv_id,
            message=user_query
        )
        
        answer = graph_response.get("answer", "") 
        
        results.append({
            "benchmark_id": sample.get("benchmark_id", ""),
            "query_id": query_id,
            "category": sample.get("category", ""),
            "difficulty": sample.get("difficulty", "unknown"),

            "question": user_query,
            "answer": answer,

            "contexts": retrieved_texts,
            "ground_truth": sample.get("detailed explanation", ""),

            "retrieved_ids": retrieved_ids,
            "ground_truth_chunk": sample.get("ground_truth_chunk_id", "")
        })
        
        time.sleep(4)

# Lưu kết quả
pd.DataFrame(results).to_json(
    "notebooks/eval_results_test3.json",
    orient="records",
    force_ascii=False
)
print("Done")

E0000 00:00:1782499718.186596   39890 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1782499718.186615   39890 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1782499718.186616   39890 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1782499718.186617   39890 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1782499718.186618   39890 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Đang đánh giá:   0%|          | 0/3 [00:00<?, ?it/s]


Đang xử lý: Bánh bao có ý nghĩa văn hóa nào trong ẩm thực Việt Nam?

Đang xử lý: Bạn có thể giải thích ý nghĩa văn hóa của bánh bao đối với nền ẩm thực Việt không?

Đang xử lý: Ý nghĩa văn hóa của bánh bao trong nền ẩm thực truyền thống Việt Nam là gì?


Đang đánh giá:  33%|███▎      | 1/3 [00:56<01:52, 56.08s/it]


Đang xử lý: Ý nghĩa văn hóa khi sử dụng lá chuối để gói bánh bột lọc là gì?

Đang xử lý: Tại sao lá chuối được dùng trong quá trình gói bánh bột lọc và ý nghĩa của điều này là gì?

Đang xử lý: Lá chuối đóng vai trò như thế nào trong việc gói bánh bột lọc từ góc độ văn hóa?


Đang đánh giá:  67%|██████▋   | 2/3 [01:41<00:49, 49.95s/it]


Đang xử lý: Việc ăn bánh căn mang ý nghĩa văn hóa như thế nào?

Đang xử lý: Ý nghĩa truyền thống của việc thưởng thức bánh căn là gì?

Đang xử lý: Câu hỏi về ý nghĩa văn hóa của việc dùng bánh căn là gì?


Đang đánh giá: 100%|██████████| 3/3 [02:24<00:00, 48.02s/it]

Done


In [4]:
df = pd.read_json("notebooks/eval_results_test3.json")

def is_hit(gt_id, retrieved_ids, k):
    gt_parts = gt_id.split(":")
    if len(gt_parts) < 5: return False
    
    gt_cat = gt_parts[0]
    gt_kw = gt_parts[1]
    gt_question = ":".join(gt_parts[4:]) # Lấy phần câu hỏi ở cuối
    
    for ret_id in retrieved_ids[:k]:
        ret_parts = ret_id.split(":")
        if len(ret_parts) < 5: continue
        
        ret_cat = ret_parts[0]
        ret_kw = ret_parts[1]
        ret_question = ":".join(ret_parts[4:])
        
        # Nếu trùng chủ đề + từ khóa + câu hỏi là được vì nhiều ảnh có thể cùng 1 câu hỏi vì thế sẽ cho hit rate = 0
        if gt_cat == ret_cat and gt_kw == ret_kw and gt_question == ret_question:
            return True
            
    return False

def hit_rate(df, k):
    hits = 0
    for _, row in df.iterrows():
        gt_id = row["ground_truth_chunk"]
        retrieved_ids = row["retrieved_ids"]
        
        if is_hit(gt_id, retrieved_ids, k):
            hits += 1
            
    return hits / len(df) if len(df) > 0 else 0

print("--- Retrieval Metrics ---")
print(f"Hit Rate @1 : {hit_rate(df, 1):.3f}")
print(f"Hit Rate @3 : {hit_rate(df, 3):.3f}")
print(f"Hit Rate @5 : {hit_rate(df, 5):.3f}")

print("\n--- Breakdown theo Category ---")
for cat, group in df.groupby("category"):
    print(f"{cat:25s} : {hit_rate(group, 5):.3f}")

print("\n--- Breakdown theo Difficulty ---")
for diff, group in df.groupby("difficulty"):
    print(f"{diff:10s} : {hit_rate(group, 5):.3f}")

--- Retrieval Metrics ---
Hit Rate @1 : 0.333
Hit Rate @3 : 0.333
Hit Rate @5 : 0.333

--- Breakdown theo Category ---
am_thuc                   : 0.333

--- Breakdown theo Difficulty ---
medium     : 0.333


Vì sử dụng dataset là 240 sample được viết lại câu hỏi sao cho không đổi ý nghĩa nên nếu tính hit rate theo doc_id sẽ cho ra kết quả rất tệ, vì các câu hỏi này đã thay đổi các từ ngữ rồi. Nên nhóm tính hit rate dựa trên khớp category và keyword để xem chatbot có truy xuất đúng chủ đề mà người dùng yêu cầu hay không

In [5]:
for _, row in df.iterrows():
    gt_id = row["ground_truth_chunk"]
    retrieved_ids = row["retrieved_ids"]
    
    if not is_hit(gt_id, retrieved_ids, 1):
        print(f"Câu hỏi: {row['question']}")
        print(f"Đáp án chuẩn : {gt_id}")
        
        if retrieved_ids:
            print(f"Bot lấy về 1 : {retrieved_ids[0]}")
            if len(retrieved_ids) > 1:
                print(f"Bot lấy về 2 : {retrieved_ids[1]}")
        else:
            print("Bot lấy về  : Không tìm thấy gì")
        print("-" * 50)

Câu hỏi: Ý nghĩa văn hóa khi sử dụng lá chuối để gói bánh bột lọc là gì?
Đáp án chuẩn : am_thuc:banh bot loc:000042:3:y nghia van hoa cua viec su dung la chuoi de goi banh bot loc la gi
Bot lấy về 1 : am_thuc:banh bot loc:000021:cultural:y nghia van hoa cua banh bot loc la gi
Bot lấy về 2 : am_thuc:banh day:000048:analysis:tai sao viec su dung la chuoi trong hinh anh nay lai quan trong
--------------------------------------------------
Câu hỏi: Tại sao lá chuối được dùng trong quá trình gói bánh bột lọc và ý nghĩa của điều này là gì?
Đáp án chuẩn : am_thuc:banh bot loc:000042:3:y nghia van hoa cua viec su dung la chuoi de goi banh bot loc la gi
Bot lấy về 1 : am_thuc:banh bot loc:000021:analysis:tai sao banh bot loc lai quan trong trong van hoa am thuc viet nam
Bot lấy về 2 : am_thuc:banh bot loc:000021:cultural:y nghia van hoa cua banh bot loc la gi
--------------------------------------------------
Câu hỏi: Lá chuối đóng vai trò như thế nào trong việc gói bánh bột lọc từ góc độ văn h

In [6]:
import pandas as pd

df = pd.read_json("notebooks/eval_results_test3.json") 

def hit_rate(df, k):
    hits = 0
    for _, row in df.iterrows():
        gt_id = str(row.get("ground_truth_chunk", ""))
        retrieved_ids = row.get("retrieved_ids", [])
        
        gt_parts = gt_id.split(":")
        if len(gt_parts) < 2: 
            continue
        gt_prefix = f"{gt_parts[0]}:{gt_parts[1]}"
        
        match = False
        for ret_id in retrieved_ids[:k]:
            ret_parts = str(ret_id).split(":")
            if len(ret_parts) >= 2:
                ret_prefix = f"{ret_parts[0]}:{ret_parts[1]}"
                
                if gt_prefix == ret_prefix:
                    match = True
                    break
        
        if match:
            hits += 1
            
    return hits / len(df) if len(df) > 0 else 0

print("--- Retrieval Metrics ---")
print(f"Hit Rate @1 : {hit_rate(df, 1):.3f}")
print(f"Hit Rate @3 : {hit_rate(df, 3):.3f}")
print(f"Hit Rate @5 : {hit_rate(df, 5):.3f}")

print("\n--- Breakdown theo Category ---")
for cat, group in df.groupby("category"):
    print(f"{cat:25s} : {hit_rate(group, 5):.3f} ({len(group)} samples)")

print("\n--- Breakdown theo Difficulty ---")
for diff, group in df.groupby("difficulty"):
    print(f"{diff:10s} : {hit_rate(group, 5):.3f} ({len(group)} samples)")

--- Retrieval Metrics ---
Hit Rate @1 : 1.000
Hit Rate @3 : 1.000
Hit Rate @5 : 1.000

--- Breakdown theo Category ---
am_thuc                   : 1.000 (9 samples)

--- Breakdown theo Difficulty ---
medium     : 1.000 (9 samples)


In [7]:
bundle = create_agent_bundle()

with open("notebooks/output_benchmark.json") as f:
    data = json.load(f)

results = []

for sample in tqdm(data, desc="Đang đánh giá"): 
    for q in sample["user_query"]:

        user_query = q["text"]
        query_id = q["query_id"]

        conv_id = f"eval_{sample.get('benchmark_id', '0')}_{query_id}" 
    
        retrieved_docs = bundle.retriever.retrieve(query=user_query)
        retrieved_texts = [chunk.document.page_content for chunk in retrieved_docs]
        retrieved_ids = [chunk.document.metadata.get("doc_id", "") for chunk in retrieved_docs]
    
        print(
            f"\n[{sample['benchmark_id']} - {query_id}] {user_query}"
        )
        try:
            graph_response = invoke_agent(
                bundle=bundle,
                user_id="evaluator_user", 
                conversation_id=conv_id,
                message=user_query
            )
            answer = graph_response.get("answer", "")
        except Exception as e:
            print(f"Lỗi ở {sample['benchmark_id']} - {query_id}: {e}")
            answer = ""
        
         
        
        results.append({
            "benchmark_id": sample.get("benchmark_id", ""),
            "query_id": query_id,
            "category": sample.get("category", ""),
            "difficulty": sample.get("difficulty", "unknown"),

            "question": user_query,
            "answer": answer,

            "contexts": retrieved_texts,
            "ground_truth": sample.get("detailed explanation", ""),

            "retrieved_ids": retrieved_ids,
            "ground_truth_chunk": sample.get("ground_truth_chunk_id", "")
        })
        
        time.sleep(4)

# Lưu kết quả
pd.DataFrame(results).to_json(
    "notebooks/eval_results_full.json",
    orient="records",
    force_ascii=False
)
print("Done")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Đang đánh giá:   0%|          | 0/240 [00:00<?, ?it/s]


[bench_0001 - q1] Bánh bao có ý nghĩa văn hóa nào trong ẩm thực Việt Nam?

[bench_0001 - q2] Bạn có thể giải thích ý nghĩa văn hóa của bánh bao đối với nền ẩm thực Việt không?

[bench_0001 - q3] Ý nghĩa văn hóa của bánh bao trong nền ẩm thực truyền thống Việt Nam là gì?


Đang đánh giá:   0%|          | 1/240 [00:49<3:17:38, 49.62s/it]


[bench_0002 - q1] Ý nghĩa văn hóa khi sử dụng lá chuối để gói bánh bột lọc là gì?

[bench_0002 - q2] Tại sao lá chuối được dùng trong quá trình gói bánh bột lọc và ý nghĩa của điều này là gì?

[bench_0002 - q3] Lá chuối đóng vai trò như thế nào trong việc gói bánh bột lọc từ góc độ văn hóa?


...


[bench_0239 - q1] Phân tích vai trò của Ca trù đối với văn hóa Việt Nam?
Lỗi ở bench_0239 - q1: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}

[bench_0239 - q2] Bạn cho rằng Ca trù có ý nghĩa gì đối với nền văn hóa Việt Nam?
Lỗi ở bench_0239 - q2: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}

[bench_0239 - q3] Theo bạn, tại sao Ca trù được coi là quan trọng trong văn hóa Việt Nam?
Lỗ

Đang đánh giá: 100%|█████████▉| 239/240 [2:32:01<00:19, 19.72s/it]


[bench_0240 - q1] Cải lương có ý nghĩa văn hóa như thế nào đối với xã hội Việt Nam?
Lỗi ở bench_0240 - q1: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}

[bench_0240 - q2] Việc nghiên cứu về vai trò của cải lương trong văn hóa cộng đồng Việt Nam mang ý nghĩa gì?
Lỗi ở bench_0240 - q2: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}

[bench_0240 - q3] Tầm ảnh hưởng của nghệ thuật cải lươ

Đang đánh giá: 100%|██████████| 240/240 [2:32:21<00:00, 38.09s/it]

Done


In [8]:
import pandas as pd

df = pd.read_json("notebooks/eval_results_full.json") 

def hit_rate(df, k):
    hits = 0
    for _, row in df.iterrows():
        gt_id = str(row.get("ground_truth_chunk", ""))
        retrieved_ids = row.get("retrieved_ids", [])
        
        gt_parts = gt_id.split(":")
        if len(gt_parts) < 2: 
            continue
        gt_prefix = f"{gt_parts[0]}:{gt_parts[1]}"
        
        match = False
        for ret_id in retrieved_ids[:k]:
            ret_parts = str(ret_id).split(":")
            if len(ret_parts) >= 2:
                ret_prefix = f"{ret_parts[0]}:{ret_parts[1]}"
                
                if gt_prefix == ret_prefix:
                    match = True
                    break
        
        if match:
            hits += 1
            
    return hits / len(df) if len(df) > 0 else 0

print("--- Retrieval Metrics ---")
print(f"Hit Rate @1 : {hit_rate(df, 1):.3f}")
print(f"Hit Rate @3 : {hit_rate(df, 3):.3f}")
print(f"Hit Rate @5 : {hit_rate(df, 5):.3f}")

print("\n--- Breakdown theo Category ---")
for cat, group in df.groupby("category"):
    print(f"{cat:25s} : {hit_rate(group, 5):.3f} ({len(group)} samples)")

print("\n--- Breakdown theo Difficulty ---")
for diff, group in df.groupby("difficulty"):
    print(f"{diff:10s} : {hit_rate(group, 5):.3f} ({len(group)} samples)")

--- Retrieval Metrics ---
Hit Rate @1 : 0.479
Hit Rate @3 : 0.600
Hit Rate @5 : 0.646

--- Breakdown theo Category ---
am_thuc                   : 0.833 (60 samples)
doi_song_hang_ngay        : 0.600 (60 samples)
giao_thong                : 0.633 (60 samples)
kien_truc                 : 0.600 (60 samples)
le_hoi                    : 0.517 (60 samples)
nhac_cu                   : 0.500 (60 samples)
phong_canh                : 0.850 (60 samples)
the_thao_truyen_thong     : 0.783 (60 samples)
thu_cong_my_nghe          : 0.667 (60 samples)
trang_phuc                : 0.567 (60 samples)
tro_choi_dan_gian         : 0.550 (60 samples)
van_hoa_dan_gian          : 0.650 (60 samples)

--- Breakdown theo Difficulty ---
hard       : 0.757 (321 samples)
medium     : 0.556 (399 samples)


In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
)
from langchain_google_genai import ChatGoogleGenerativeAI
import os

eval_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)
eval_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base"
)
df_results = pd.read_json("notebooks/eval_results_full.json")

eval_dataset = Dataset.from_pandas(df_results[["question", "answer", "contexts", "ground_truth"]])


metrics = [
    faithfulness,       
    answer_relevancy,   
]

# Hàm evaluate của RAGAS
score = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=eval_llm,
    embeddings=eval_embeddings
)

print("\nĐIỂM SỐ TỔNG QUAN")
print(score)

df_ragas_details = score.to_pandas()

df_ragas_details.to_csv("notebooks/ragas_full_score.csv", index=False)
print("\nĐã lưu kết quả chi tiết ra file: notebooks/ragas_full_score.csv")

/tmp/ipykernel_75064/177023869.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_75064/177023869.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1440 [00:00<?, ?it/s]

Exception raised in Job[0]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[15]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[12]: TimeoutError()


KeyboardInterrupt: 

Exception raised in Job[29]: ChatGoogleGenerativeAIError(Error calling model 'gemini-2.5-flash' (UNAUTHENTICATED): 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'Request had invalid authentication credentials. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project.', 'status': 'UNAUTHENTICATED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'ACCESS_TOKEN_TYPE_UNSUPPORTED', 'metadata': {'method': 'google.ai.generativelanguage.v1beta.GenerativeService.GenerateContent', 'service': 'generativelanguage.googleapis.com'}}]}})
Exception raised in Job[32]: AssertionError(LLM is not set)
Exception raised in Job[33]: AssertionError(LLM is not set)
Exception raised in Job[34]: AssertionError(LLM is not set)
Exception raised in Job[35]: AssertionError(LLM is not set)
Exception raised in Job[36]: AssertionError(LLM is not set)
Exception raised in Job[37]: A

In [3]:
import pandas as pd
import os

# Kiểm tra checkpoint đã lưu
if os.path.exists("notebooks/ragas_checkpoint.csv"):
    df_check = pd.read_csv("notebooks/ragas_checkpoint.csv")
    print(f"Đã lưu: {len(df_check)} samples")
    print(f"Faithfulness    : {df_check['faithfulness'].mean():.3f}")
    print(f"Answer Relevancy: {df_check['answer_relevancy'].mean():.3f}")
else:
    print("Không có checkpoint")
    
# Kiểm tra full score
if os.path.exists("notebooks/ragas_full_score.csv"):
    df_full = pd.read_csv("notebooks/ragas_full_score.csv")
    print(f"\nFull: {len(df_full)} samples")
    print(df_full[["faithfulness","answer_relevancy"]].describe())

Không có checkpoint


In [3]:
# Chạy cell này trước để test
from langchain_google_genai import ChatGoogleGenerativeAI

eval_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_retries=3,
    timeout=60,
)

# Test 1 call đơn giản
response = eval_llm.invoke("Xin chào, trả lời ngắn thôi")
print(response.content)
print("✅ API còn quota")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_requests_per_model_per_day, limit: 10000, model: gemini-2.5-flash\nPlease retry in 23h32m27.601343966s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_requests_per_model_per_day', 'quotaId': 'GenerateRequestsPerDayPerProjectPerMo

In [ ]:
import os
import pandas as pd
import time
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings

os.environ["GOOGLE_API_KEY"] = "KEY_CỦA_BẠN"

eval_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_retries=3,
    timeout=60,
)

eval_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base"
)

# ── Dùng thẳng 720 samples ─────────────────────────────
df_results = pd.read_json("notebooks/eval_results_full.json")
print(f"Tổng samples: {len(df_results)}")  # 720

CHECKPOINT = "notebooks/ragas_checkpoint.csv"
BATCH_SIZE = 10
SLEEP_SEC  = 30   

if os.path.exists(CHECKPOINT):
    df_done    = pd.read_csv(CHECKPOINT)
    done_count = len(df_done)
    all_scores = [df_done]
    print(f"Resume từ sample {done_count}")
else:
    all_scores = []
    done_count = 0
    print("Bắt đầu từ đầu")

df_remaining = df_results.iloc[done_count:].reset_index(drop=True)
total        = len(df_remaining)
print(f"Còn lại: {total} samples\n")

for i in range(0, total, BATCH_SIZE):
    batch     = df_remaining.iloc[i:i+BATCH_SIZE].copy()
    batch_num = (done_count + i) // BATCH_SIZE + 1
    n_end     = done_count + i + len(batch)

    print(f"{'─'*50}")
    print(f"Batch {batch_num}: samples {done_count+i+1} → {n_end}")

    try:
        dataset = Dataset.from_pandas(
            batch[["question", "answer",
                   "contexts", "ground_truth"]]
        )
        score = evaluate(
            dataset=dataset,
            metrics=[faithfulness, answer_relevancy],
            llm=eval_llm,
            embeddings=eval_embeddings,
            raise_exceptions=False,
        )
        df_batch = score.to_pandas()
        all_scores.append(df_batch)

        pd.concat(all_scores, ignore_index=True).to_csv(
            CHECKPOINT, index=False
        )

        faith = df_batch["faithfulness"].mean()
        rel   = df_batch["answer_relevancy"].mean()
        print(f"faith={faith:.3f} | relevancy={rel:.3f}")

    except Exception as e:
        print(f"Lỗi: {e}")
        print(f"Nghỉ 60s rồi thử lại")
        time.sleep(60)
        try:
            score = evaluate(
                dataset=dataset,
                metrics=[faithfulness, answer_relevancy],
                llm=eval_llm,
                embeddings=eval_embeddings,
                raise_exceptions=False,
            )
            df_batch = score.to_pandas()
            all_scores.append(df_batch)
            pd.concat(all_scores, ignore_index=True).to_csv(
                CHECKPOINT, index=False
            )
            print(f"Retry OK")
        except Exception as e2:
            print(f"Bỏ qua batch này: {e2}")

    if i + BATCH_SIZE < total:
        print(f"Nghỉ {SLEEP_SEC}s...")
        time.sleep(SLEEP_SEC)

if all_scores:
    df_final = pd.concat(all_scores, ignore_index=True)
    df_final.to_csv("notebooks/ragas_full_score.csv", index=False)

    print(f"\n{'═'*50}")
    print(f"TỔNG KẾT ({len(df_final)}/720 samples)")
    print(f"{'═'*50}")
    print(f"Faithfulness    : {df_final['faithfulness'].mean():.3f}")
    print(f"Answer Relevancy: {df_final['answer_relevancy'].mean():.3f}")

    # Breakdown theo category
    df_merged = df_results.iloc[:len(df_final)].copy()
    df_merged["faithfulness"]    = df_final["faithfulness"].values
    df_merged["answer_relevancy"] = df_final["answer_relevancy"].values

    print(f"\n── Theo Category ──")
    for cat, g in df_merged.groupby("category"):
        print(f"  {cat:25s}: "
              f"faith={g['faithfulness'].mean():.3f} | "
              f"rel={g['answer_relevancy'].mean():.3f}")

In [ ]:
print(df_ragas_details[["faithfulness","answer_relevancy"]].mean())

In [ ]:
merged = pd.concat(
    [
        df_results[["category","difficulty"]],
        df_ragas_details[["faithfulness","answer_relevancy"]],
    ],
    axis=1
)

print(merged.groupby("category")[["faithfulness","answer_relevancy"]].mean())

print(merged.groupby("difficulty")[["faithfulness","answer_relevancy"]].mean())